In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stable_baselines3 import PPO, A2C, DDPG, SAC, TD3
from finrl.config import INDICATORS


INITIAL_CAPITAL = 100_000
RISK_FREE_RATE = 0.0
TRADING_DAYS = 252

PROJECT_ROOT = Path(r"C:\Users\fabia\Desktop\FinRl\pro-finrl\src\notebooks")
MODEL_DIR = PROJECT_ROOT / "trained_models"

RESULT_DIR = PROJECT_ROOT / "results" / "benchmarks"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
#Testdaten laden und  an das tatsächliche Datenformat anpassen

test_data = pd.read_csv(r"C:\Users\fabia\Desktop\FinRl\pro-finrl\src\data\trade.csv") ###Pfad anpassen


test_data["date"] = pd.to_datetime(test_data["date"])
test_data = test_data.sort_values(["date", "tic"])

test_data.head()

,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2020-09-01,A,96.983643,97.146687,96.149228,96.791819,1159700.0,1.0,1.401527,96.510749,91.996429,63.493863,227.721755,23.941270,93.730402,89.642066,26.120001,419.67357
1,2020-09-01,AAPL,130.038879,130.639755,126.501533,128.662706,151948100.0,1.0,7.266951,129.203013,101.303442,76.649730,152.026260,72.627209,108.659106,98.615379,26.120001,419.67357
2,2020-09-01,ABT,95.254753,97.507608,94.387581,97.382449,7963100.0,1.0,2.360313,98.475653,85.444245,59.907787,152.646862,33.261999,91.258612,86.690473,26.120001,419.67357
3,2020-09-01,ACGL,30.048393,30.162500,29.668034,29.734597,930100.0,1.0,0.276094,31.424173,29.147112,52.567445,6.978326,19.005452,29.922875,29.017144,26.120001,419.67357
4,2020-09-01,ACN,218.043655,218.043655,215.481849,216.653744,1914500.0,1.0,4.417208,221.736901,205.010776,64.972031,100.148009,41.626485,210.107847,201.085200,26.120001,419.67357


In [3]:
MODEL_CONFIG = {
    "ppo": {
        "algorithm": PPO,
        "path": MODEL_DIR / "agent_ppo.zip",  ###Pfad anpassen
    },
    "a2c": {
        "algorithm": A2C,
        "path": MODEL_DIR / "agent_a2c.zip",  ###Pfad anpassen
    },
    "ddpg": {
        "algorithm": DDPG,
        "path": MODEL_DIR / "agent_ddpg.zip", ###Pfad anpassen
    },
    "sac": {
        "algorithm": SAC,
        "path": MODEL_DIR / "agent_sac.zip", ###Pfad anpassen
    },
    # Weitere Modelle hier ergänzen
}

In [4]:
##KI generierter Inhalt!!!


"""
Trade-Republic-Kosten für das FinRL StockTradingEnv (Stand: September 2026).

Berücksichtigt nur:
  - Ordergebühr: 1 € (Bestpreis) bzw. 2 € (Direktpreis) fix pro Order
  - Transaktionskosten: geschätzter Spread in Prozent
"""
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv


class TradeRepublicStockTradingEnv(StockTradingEnv):
    def __init__(self, df, *, order_fee=1.0, spread_pct=0.0005, **kwargs):
        self.order_fee = order_fee      # 1.0 = Bestpreis, 2.0 = Direktpreis
        self.spread_pct = spread_pct    # z. B. 0.0005 = 0,05 %

        # Prozentuale Kosten des Basis-Envs abschalten
        n = kwargs["stock_dim"]
        kwargs["buy_cost_pct"] = [0.0] * n
        kwargs["sell_cost_pct"] = [0.0] * n
        super().__init__(df=df, **kwargs)

    def _turbulent(self):
        return (self.turbulence_threshold is not None
                and self.turbulence >= self.turbulence_threshold)

    def _sell_stock(self, index, action):
        price = self.state[1 + index]
        held = self.state[1 + self.stock_dim + index]
        if price <= 0 or held <= 0:
            return 0

        # Bei Turbulenz alles verkaufen (wie im Basis-Env)
        shares = int(held if self._turbulent() else min(abs(action), held))
        proceeds = price * shares * (1 - self.spread_pct)
        if shares <= 0 or proceeds <= self.order_fee:
            return 0

        self.state[0] += proceeds - self.order_fee
        self.state[1 + self.stock_dim + index] -= shares
        self.cost += price * shares * self.spread_pct + self.order_fee
        self.trades += 1
        return shares

    def _buy_stock(self, index, action):
        price = self.state[1 + index]
        if price <= 0 or self._turbulent():
            return 0

        ask = price * (1 + self.spread_pct)
        affordable = int((self.state[0] - self.order_fee) // ask)
        shares = int(min(affordable, action))
        if shares <= 0:
            return 0

        self.state[0] -= ask * shares + self.order_fee
        self.state[1 + self.stock_dim + index] += shares
        self.cost += price * shares * self.spread_pct + self.order_fee
        self.trades += 1
        return shares

In [5]:
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv

TECH_INDICATORS = INDICATORS


def make_test_env(data: pd.DataFrame):
    data = data.copy()

    if "close" not in data.columns and "adjcp" in data.columns:
        data["close"] = data["adjcp"]

    required_columns = {"date", "tic", "close", *TECH_INDICATORS}
    missing_columns = required_columns - set(data.columns)

    if missing_columns:
        raise ValueError(
            f"Fehlende Spalten in test_data: {sorted(missing_columns)}"
        )

    data["date"] = pd.to_datetime(data["date"])
    data = data.sort_values(["date", "tic"]).reset_index(drop=True)

    # Jeder Handelstag erhält einen gemeinsamen Index für FinRL.
    data["day"] = pd.factorize(data["date"])[0]
    data = data.set_index("day", drop=False)

    stock_dim = int(data["tic"].nunique())
    state_space = int(
        1 + 2 * stock_dim + len(TECH_INDICATORS) * stock_dim      
    )

    return TradeRepublicStockTradingEnv(
        df=data,
        hmax=100,
        initial_amount=INITIAL_CAPITAL,
        num_stock_shares=[0] * stock_dim,
        buy_cost_pct=[0.001] * stock_dim,
        sell_cost_pct=[0.001] * stock_dim,
        state_space=state_space,
        action_space=stock_dim,
        stock_dim=stock_dim,
        tech_indicator_list=TECH_INDICATORS,
        reward_scaling=1e-4,
    )

In [6]:
test_env = make_test_env(test_data)
reset_result = test_env.reset()

observation = (
    reset_result[0]
    if isinstance(reset_result, tuple)
    else reset_result
)

observation = np.asarray(observation, dtype=np.float32)

print("Umgebung erfolgreich erstellt.")
print("Observation:", observation.shape)

Umgebung erfolgreich erstellt.
Observation: (4141,)


In [7]:
def create_price_matrix(data: pd.DataFrame) -> pd.DataFrame:
    price_column = "adjcp" if "adjcp" in data.columns else "close"

    prices = data.pivot(
        index="date",
        columns="tic",
        values=price_column,
    ).sort_index()

    return prices.ffill().dropna()


def buy_and_hold(prices: pd.DataFrame) -> pd.Series:
    """Gleichgewichtetes Portfolio mit konstanten Stückzahlen."""
    weights = np.ones(prices.shape[1]) / prices.shape[1]
    initial_prices = prices.iloc[0]
    shares = (INITIAL_CAPITAL * weights) / initial_prices
    portfolio_value = prices.mul(shares, axis=1).sum(axis=1)

    return portfolio_value.rename("Buy_and_Hold")


def equal_weight_rebalanced(prices: pd.DataFrame) -> pd.Series:
    """Täglich gleichgewichtetes Portfolio."""
    returns = prices.pct_change().fillna(0)
    portfolio_returns = returns.mean(axis=1)

    portfolio_value = INITIAL_CAPITAL * (1 + portfolio_returns).cumprod()

    return portfolio_value.rename("Equal_Weight")


def market_index(prices: pd.DataFrame) -> pd.Series:
    """Preisgewichteter Index, normiert auf das Startkapital."""
    index_value = (
        prices.sum(axis=1)
        / prices.iloc[0].sum()
        * INITIAL_CAPITAL
    )

    return index_value.rename("Market_Index")


prices = create_price_matrix(test_data)

benchmark_values = pd.concat(
    [
        buy_and_hold(prices),
        equal_weight_rebalanced(prices),
        market_index(prices),
    ],
    axis=1,
)

benchmark_values.head()

,Buy_and_Hold,Equal_Weight,Market_Index
date,,,
2020-09-01,100000.000000,100000.000000,100000.000000
2020-09-02,101866.637340,101866.637340,101801.329242
2020-09-03,99348.140877,99350.092844,98523.542903
2020-09-04,99030.775222,99016.192364,97587.298150
2020-09-08,96919.545587,96895.580515,95769.054653


In [8]:
def get_portfolio_value(info, fallback_value, env=None):
    """
    Unterschiedliche FinRL-Umgebungen verwenden unterschiedliche Info-Schlüssel.
    """
    possible_keys = [
        "portfolio_value",
        "total_asset",
        "account_value",
        "asset_value",
        "total_assets",
    ]

    #KI generiert aufgrund von technischen Problemen den Code nicht korrekt.
    
    for key in possible_keys:
        if key in info:
            return float(info[key])

    tracked_env = env
    while tracked_env is not None and hasattr(tracked_env, "envs"):
        envs = tracked_env.envs
        if not envs:
            break
        tracked_env = envs[0]

    if tracked_env is not None and hasattr(tracked_env, "unwrapped"):
        tracked_env = tracked_env.unwrapped

    if tracked_env is not None and getattr(tracked_env, "asset_memory", None):
        return float(tracked_env.asset_memory[-1])

    return fallback_value


def evaluate_model(model_name: str, config: dict, data: pd.DataFrame):
    env = make_test_env(data)

    model = config["algorithm"].load(
        str(config["path"]),
        env=env,
        device="auto",
    )

    reset_result = env.reset()

    if isinstance(reset_result, tuple):
        observation = reset_result[0]
    else:
        observation = reset_result

    portfolio_values = [INITIAL_CAPITAL]
    dates = sorted(data["date"].unique())

    terminated = False
    truncated = False
    step_index = 0

    while not (terminated or truncated):
        action, _ = model.predict(
            observation,
            deterministic=True,
        )

        step_result = env.step(action)

        if len(step_result) == 5:
            observation, reward, terminated, truncated, info = step_result
        else:
            observation, reward, done, info = step_result
            terminated = done
            truncated = False

        previous_value = portfolio_values[-1]
        current_value = get_portfolio_value(info, previous_value, env)
        portfolio_values.append(current_value)

        step_index += 1

        if step_index >= len(dates):
            break

    result_dates = dates[:len(portfolio_values)]

    return pd.Series(
        portfolio_values[:len(result_dates)],
        index=pd.to_datetime(result_dates),
        name=model_name,
    )

In [9]:
model_values = {}

for model_name, config in MODEL_CONFIG.items():
    model_path = Path(config["path"])

    if not model_path.exists():
        print(f"Übersprungen, Datei nicht gefunden: {model_path}")
        continue

    print(f"Evaluierung: {model_name}")

    try:
        model_values[model_name] = evaluate_model(
            model_name=model_name,
            config=config,
            data=test_data,
        )
    except Exception as exc:
        print(f"Fehler bei {model_name}: {type(exc).__name__}: {exc}")


portfolio_frames = [benchmark_values]

if model_values:
    model_results = pd.concat(
        list(model_values.values()),
        axis=1,
    )
    portfolio_frames.append(model_results)
else:
    print("Keine trainierten Modelle konnten evaluiert werden.")

all_portfolio_values = pd.concat(
    portfolio_frames,
    axis=1,
).sort_index()

all_portfolio_values = (
    all_portfolio_values
    .apply(pd.to_numeric, errors="coerce")
    .ffill()
    .dropna()
)

print("Ausgewertete Strategien:")
print(all_portfolio_values.columns.tolist())

all_portfolio_values.head()

Evaluierung: ppo
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Evaluierung: a2c
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Übersprungen, Datei nicht gefunden: C:\Users\fabia\Desktop\FinRl\pro-finrl\src\notebooks\trained_models\agent_ddpg.zip
Evaluierung: sac
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Ausgewertete Strategien:
['Buy_and_Hold', 'Equal_Weight', 'Market_Index', 'ppo', 'a2c', 'sac']


,Buy_and_Hold,Equal_Weight,Market_Index,ppo,a2c,sac
2020-09-01,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
2020-09-02,101866.637340,101866.637340,101801.329242,100175.684292,101850.808606,101656.709896
2020-09-03,99348.140877,99350.092844,98523.542903,99046.265927,98326.831376,97426.415676
2020-09-04,99030.775222,99016.192364,97587.298150,98465.808040,97178.631373,95754.734915
2020-09-08,96919.545587,96895.580515,95769.054653,96946.590681,94766.312279,91463.458987


In [10]:
for model_name, config in MODEL_CONFIG.items():
    print(model_name, "->", config["path"], "->", Path(config["path"]).exists())

ppo -> C:\Users\fabia\Desktop\FinRl\pro-finrl\src\notebooks\trained_models\agent_ppo.zip -> True
a2c -> C:\Users\fabia\Desktop\FinRl\pro-finrl\src\notebooks\trained_models\agent_a2c.zip -> True
ddpg -> C:\Users\fabia\Desktop\FinRl\pro-finrl\src\notebooks\trained_models\agent_ddpg.zip -> False
sac -> C:\Users\fabia\Desktop\FinRl\pro-finrl\src\notebooks\trained_models\agent_sac.zip -> True


In [11]:
def calculate_metrics(portfolio_values: pd.Series) -> dict:
    returns = portfolio_values.pct_change().dropna()

    if len(returns) == 0:
        return {}

    cumulative_return = (
        portfolio_values.iloc[-1] / portfolio_values.iloc[0]
    ) - 1

    years = len(returns) / TRADING_DAYS
    annual_return = (
        (portfolio_values.iloc[-1] / portfolio_values.iloc[0]) ** (1 / years) - 1
        if years > 0
        else np.nan
    )

    annual_volatility = returns.std() * np.sqrt(TRADING_DAYS)

    sharpe_ratio = (
        (returns.mean() / returns.std()) * np.sqrt(TRADING_DAYS)
        if returns.std() != 0
        else np.nan
    )

    downside_returns = returns[returns < 0]
    downside_volatility = downside_returns.std() * np.sqrt(TRADING_DAYS)

    sortino_ratio = (
        (returns.mean() * TRADING_DAYS) / downside_volatility
        if downside_volatility != 0
        else np.nan
    )

    running_max = portfolio_values.cummax()
    drawdown = portfolio_values / running_max - 1
    max_drawdown = drawdown.min()

    calmar_ratio = (
        annual_return / abs(max_drawdown)
        if max_drawdown != 0
        else np.nan
    )

    return {
        "final_value": portfolio_values.iloc[-1],
        "cumulative_return": cumulative_return,
        "annual_return": annual_return,
        "annual_volatility": annual_volatility,
        "sharpe_ratio": sharpe_ratio,
        "sortino_ratio": sortino_ratio,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar_ratio,
    }


metrics = pd.DataFrame.from_dict(
    {
        strategy: calculate_metrics(all_portfolio_values[strategy])
        for strategy in all_portfolio_values.columns
    },
    orient="index",
)

metrics = metrics.sort_values(
    by="sharpe_ratio",
    ascending=False,
)

metrics

,final_value,cumulative_return,annual_return,annual_volatility,sharpe_ratio,sortino_ratio,max_drawdown,calmar_ratio
Equal_Weight,262200.647860,1.622006,0.174782,0.158882,1.093485,1.590449,-0.186786,0.935736
Buy_and_Hold,273283.238655,1.732832,0.182938,0.168178,1.083277,1.579414,-0.187227,0.977091
ppo,265707.082359,1.657071,0.177393,0.185551,0.972994,1.454606,-0.251334,0.705806
Market_Index,205247.113735,1.052471,0.127676,0.158086,0.839201,1.241515,-0.233630,0.546487
a2c,190220.045685,0.902200,0.113438,0.185820,0.671169,1.014745,-0.299083,0.379288
sac,153462.683082,0.534627,0.074194,0.222360,0.433356,0.586754,-0.399768,0.185592


In [12]:
# Efficient Frontier aus zufälligen Mischungen der vorhandenen Strategien
strategy_returns = all_portfolio_values.pct_change().dropna()
annual_returns = strategy_returns.mean() * TRADING_DAYS
annual_covariance = strategy_returns.cov() * TRADING_DAYS

rng = np.random.default_rng(SEED)
num_portfolios = 10_000
num_strategies = len(all_portfolio_values.columns)

weights = rng.dirichlet(np.ones(num_strategies), size=num_portfolios)
portfolio_returns = weights @ annual_returns.to_numpy()
portfolio_volatility = np.sqrt(
    np.einsum("ij,jk,ik->i", weights, annual_covariance.to_numpy(), weights)
)
portfolio_sharpe = np.divide(
    portfolio_returns - RISK_FREE_RATE,
    portfolio_volatility,
    out=np.full_like(portfolio_returns, np.nan),
    where=portfolio_volatility != 0,
)

max_sharpe_index = int(np.nanargmax(portfolio_sharpe))
min_volatility_index = int(np.nanargmin(portfolio_volatility))
strategy_labels = globals().get("agent_labels", {})
frontier_colors = {
    "Buy_and_Hold": "#0072B2",
    "Equal_Weight": "#E69F00",
    "Market_Index": "#009E73",
    "ppo": "#D55E00",
    "a2c": "#CC79A7",
    "ddpg": "#56B4E9",
    "td3": "#F0E442",
    "sac": "#000000",
}

plt.figure(figsize=(12, 7))
scatter = plt.scatter(
    portfolio_volatility,
    portfolio_returns,
    c=portfolio_sharpe,
    cmap="viridis",
    s=8,
    alpha=0.35,
    label="Zufällige Portfolios",
)
plt.colorbar(scatter, label="Sharpe Ratio")

plt.scatter(
    portfolio_volatility[max_sharpe_index],
    portfolio_returns[max_sharpe_index],
    color="red",
    marker="*",
    s=220,
    label="Maximale Sharpe Ratio",
    zorder=3,
)
plt.scatter(
    portfolio_volatility[min_volatility_index],
    portfolio_returns[min_volatility_index],
    color="black",
    marker="X",
    s=110,
    label="Minimale Volatilität",
    zorder=3,
)

strategy_volatility = strategy_returns.std() * np.sqrt(TRADING_DAYS)
for strategy in all_portfolio_values.columns:
    plt.scatter(
        strategy_volatility[strategy],
        annual_returns[strategy],
        color=frontier_colors.get(strategy, "#666666"),
        edgecolors="black",
        s=90,
        marker="o",
        label=strategy_labels.get(strategy, strategy),
        zorder=4,
    )
    plt.annotate(
        strategy_labels.get(strategy, strategy),
        (strategy_volatility[strategy], annual_returns[strategy]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )

plt.title("Efficient Frontier der Strategien")
plt.xlabel("Annualisierte Volatilität")
plt.ylabel("Annualisierte Rendite")
plt.grid(True, alpha=0.3)
plt.legend(loc="best")
plt.tight_layout()
plt.savefig(
    RESULT_DIR / "efficient_frontier.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

print(
    "Maximale Sharpe Ratio:",
    round(portfolio_sharpe[max_sharpe_index], 3),
)
print(
    "Minimale Volatilität:",
    round(portfolio_volatility[min_volatility_index], 3),
)

Maximale Sharpe Ratio: 1.092
Minimale Volatilität: 0.158


C:\Users\fabia\AppData\Local\Temp\ipykernel_15304\1990932924.py:98: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
agent_labels = {
    "Buy_and_Hold": "Buy & Hold",
    "Equal_Weight": "Equal Weight",
    "Market_Index": "Market Index",
    "ppo": "PPO-Agent",
    "a2c": "A2C-Agent",
    "ddpg": "DDPG-Agent",
    "td3": "TD3-Agent",
    "sac": "SAC-Agent",
}

strategy_styles = {
    "Buy_and_Hold": {"color": "#0072B2", "linestyle": "-", "marker": "o"},
    "Equal_Weight": {"color": "#E69F00", "linestyle": "--", "marker": "s"},
    "Market_Index": {"color": "#009E73", "linestyle": ":", "marker": "D"},
    "ppo": {"color": "#D55E00", "linestyle": "-.", "marker": "^"},
    "a2c": {"color": "#CC79A7", "linestyle": "-", "marker": "v"},
    "ddpg": {"color": "#56B4E9", "linestyle": "--", "marker": "P"},
    "td3": {"color": "#F0E442", "linestyle": ":", "marker": "X"},
    "sac": {"color": "#000000", "linestyle": "-.", "marker": "*"},
}

plt.figure(figsize=(15, 7))

for column in all_portfolio_values.columns:
    normalized_values = (
        all_portfolio_values[column]
        / all_portfolio_values[column].iloc[0]
        * 100
    )

    style = strategy_styles.get(
        column,
        {"color": "#666666", "linestyle": "-", "marker": "o"},
    )
    label = agent_labels.get(column, column)
    plt.plot(
        normalized_values,
        label=label,
        linewidth=2.2,
        color=style["color"],
        linestyle=style["linestyle"],
        marker=style["marker"],
        markersize=5,
        markevery=max(len(normalized_values) // 10, 1),
    )

plt.title("Portfolioentwicklung im Tradezeitraum")
plt.xlabel("Datum")
plt.ylabel("Indexierter Portfoliowert")
plt.legend(
    title="Strategien und Agenten",
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    RESULT_DIR / "portfolio_comparison.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

C:\Users\fabia\AppData\Local\Temp\ipykernel_15304\2543614938.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
plt.figure(figsize=(15, 7))

for column in all_portfolio_values.columns:
    running_max = all_portfolio_values[column].cummax()
    drawdown = all_portfolio_values[column] / running_max - 1
    style = strategy_styles.get(
        column,
        {"color": "#666666", "linestyle": "-", "marker": "o"},
    )

    plt.plot(
        drawdown,
        label=agent_labels.get(column, column),
        linewidth=2.2,
        color=style["color"],
        linestyle=style["linestyle"],
        marker=style["marker"],
        markersize=5,
        markevery=max(len(drawdown) // 10, 1),
    )

plt.title("Drawdown-Vergleich")
plt.xlabel("Datum")
plt.ylabel("Drawdown")
plt.legend(
    title="Strategien und Agenten",
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    RESULT_DIR / "drawdown_comparison.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

C:\Users\fabia\AppData\Local\Temp\ipykernel_15304\409209900.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
metrics.to_csv(RESULT_DIR / "performance_metrics.csv")
all_portfolio_values.to_csv(RESULT_DIR / "portfolio_values.csv")

print(f"Ergebnisse gespeichert unter: {RESULT_DIR}")

Ergebnisse gespeichert unter: C:\Users\fabia\Desktop\FinRl\pro-finrl\src\notebooks\results\benchmarks
